In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV

# 1. Separate features and target
data = {
    "area_sqft": [750, 900, 1200, 1500, 1800, 2100, 2400, 2700, 3000, 3500],
    "distance_school_km": [1.2, 2.1, 1.0, 3.5, 2.8, 4.0, 3.0, 5.2, 4.5, 6.0],
    "house_price": [45, 52, 65, 72, 85, 95, 110, 125, 140, 165]
}
df = pd.DataFrame(data)
X = df[["area_sqft", "distance_school_km"]]
y = df["house_price"]

# 2. Create a Pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsRegressor())
])

# 3 & 4. GridSearchCV with 5-fold CV
param_grid = {'model__n_neighbors': [1, 2, 3, 4, 5]}
grid = GridSearchCV(pipe, param_grid, cv=5)
grid.fit(X, y)

# 5. Print results
print(f"Best Parameter: {grid.best_params_}")
print(f"Best Score: {grid.best_score_}")

# 6. Convert to DataFrame
results_df = pd.DataFrame(grid.cv_results_)
print(results_df[['param_model__n_neighbors', 'mean_test_score', 'rank_test_score']])

Best Parameter: {'model__n_neighbors': 1}
Best Score: -12.544907029478457
   param_model__n_neighbors  mean_test_score  rank_test_score
0                         1       -12.544907                1
1                         2       -16.967846                2
2                         3       -19.184854                3
3                         4       -23.987305                4
4                         5       -29.279427                5


In [2]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, precision_score, recall_score, confusion_matrix

# 1. Split data
data = {
    "transaction_amount": [200, 4500, 120, 8000, 90, 15000, 300, 700, 20000, 150],
    "transaction_time": [10, 2, 14, 3, 16, 1, 11, 13, 2, 17],
    "previous_failed_attempts": [0, 3, 0, 4, 0, 5, 1, 0, 6, 0],
    "fraud": [0, 1, 0, 1, 0, 1, 0, 0, 1, 0]
}
df = pd.DataFrame(data)
X = df.drop("fraud", axis=1)
y = df["fraud"]

# 2. Create Pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced'))
])

# 3. Custom Scorer (minimum of precision and recall)
def min_precision_recall(y_true, y_pred):
    return min(precision_score(y_true, y_pred, zero_division=0), recall_score(y_true, y_pred, zero_division=0))

custom_scorer = make_scorer(min_precision_recall)

# 4. GridSearchCV
param_grid = {'model__C': [0.01, 0.1, 1, 10]}
grid = GridSearchCV(pipe, param_grid, cv=3, scoring=custom_scorer) # using cv=3 due to small dataset
grid.fit(X, y)

# 5 & 6. Print Results
best_model = grid.best_estimator_
y_pred = best_model.predict(X)

print(f"Best C: {grid.best_params_['model__C']}")
print(f"Precision: {precision_score(y, y_pred)}")
print(f"Recall: {recall_score(y, y_pred)}")
print(f"Confusion Matrix:\n{confusion_matrix(y, y_pred)}")

Best C: 0.01
Precision: 1.0
Recall: 1.0
Confusion Matrix:
[[6 0]
 [0 4]]


In [3]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# Data Setup
data = {
    "study_hours": [1, 2, 3, 4, 5, 6, 7, 8],
    "attendance": [40, 50, 55, 60, 70, 75, 85, 90],
    "learning_mode": ["online", "offline", "online", "offline", "hybrid", "hybrid", "offline", "online"],
    "passed": [0, 0, 0, 1, 1, 1, 1, 1]
}
df = pd.DataFrame(data)
X = df.drop("passed", axis=1)
y = df["passed"]

# 1. Separate Columns
num_cols = ["study_hours", "attendance"]
cat_cols = ["learning_mode"]

# 2, 3, & 4. Column Transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('poly', PolynomialFeatures(degree=2)),
            ('scaler', StandardScaler())
        ]), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

# 5. Train Pipeline
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression())
])
pipe.fit(X, y)

# 6. Predict for New Student
new_student = pd.DataFrame({
    "study_hours": [5],
    "attendance": [72],
    "learning_mode": ["online"]
})

prediction = pipe.predict(new_student)
print(f"Prediction (1 = Passed, 0 = Failed): {prediction[0]}")

Prediction (1 = Passed, 0 = Failed): 1
